# Policy to Pipeline Radar: prototype notebook

Course prototype for MPIM-6750 (Module 5). The Radar pulls public federal signals, normalizes them into one schema, and places each on a three-tier evidence ladder so a capture manager can see likely opportunities before a solicitation posts.

**Public sources only:** Federal Register, USAspending, and SAM.gov. No internal or proprietary data.

**Run order:** top to bottom (Run > Run All Cells). Raw API responses land in `data/raw/` before anything is changed.

## 0. Setup

One config cell controls what the Radar looks for. The SAM.gov key is read from the `SAM_API_KEY` environment variable and is never printed or saved.

In [1]:
import os
import json
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ---- What the Radar looks for ----
FOCUS = "zero trust"
AGENCY_NAME = "Department of Defense"          # USAspending awarding agency
FR_AGENCY_SLUGS = ["defense-department"]        # Federal Register agency filter
SAM_DEPT_PREFIX = "DEPT OF DEFENSE"             # SAM.gov org path starts with this for DoD

# The Federal Register rarely uses the phrase "zero trust" (0 DoD documents in the last 90 days
# or the last year, checked 2026-10-11), so the policy search widens to the rules that drive the work.
FR_TERM = '"zero trust" | cybersecurity | CMMC'
AWARD_KEYWORDS = ["zero trust"]
SAM_TITLE = "zero trust"                        # SAM.gov searches titles only

# ---- Time windows ----
LOOKBACK_DAYS = 90
# DoD contract data reaches FPDS and USAspending on a 90-day delay, so a 90-day award window
# comes back nearly empty (0 DoD zero trust awards, checked 2026-10-11). Awards look back a year.
AWARD_LOOKBACK_DAYS = 365

TODAY = date.today()
START = TODAY - timedelta(days=LOOKBACK_DAYS)
AWARD_START = TODAY - timedelta(days=AWARD_LOOKBACK_DAYS)

# ---- Paths and request settings ----
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TIMEOUT = 30
MAX_PAGES = 10                                  # safety cap on paging loops
HEADERS = {"User-Agent": "policy-to-pipeline-radar (course prototype)"}
SAM_REFRESH = False                             # SAM.gov caps daily requests; reuse today's pull unless True

SAM_API_KEY = os.environ.get("SAM_API_KEY")     # never printed, never written to disk

pd.set_option("display.max_colwidth", None)     # show full status notes and titles

print(f"Signal window: {START} to {TODAY} ({LOOKBACK_DAYS} days)")
print(f"Award window:  {AWARD_START} to {TODAY} ({AWARD_LOOKBACK_DAYS} days)")
print("SAM.gov key:", "found" if SAM_API_KEY else "missing, SAM.gov will be skipped")

Signal window: 2026-07-13 to 2026-10-11 (90 days)
Award window:  2025-10-11 to 2026-10-11 (365 days)
SAM.gov key: found


## 1. Ingest

One function per source. Each one saves the raw response to `data/raw/` before any filtering or reshaping, then returns the records. If a source fails, the run prints a status line and moves on to the next source instead of stopping.

In [2]:
STATUS = []   # one row per source per run


def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def redact(text):
    """Strip the SAM.gov key from any string before it is printed or saved."""
    text = str(text)
    if SAM_API_KEY:
        text = text.replace(SAM_API_KEY, "***")
    return text


def save_raw(source, payload):
    """Write the untouched API response to data/raw/<source>_<YYYYMMDD>.json."""
    path = RAW_DIR / f"{source}_{TODAY:%Y%m%d}.json"
    path.write_text(redact(json.dumps(payload, indent=2)), encoding="utf-8")
    return path


def log_status(source, ok, n, note):
    label = "OK" if ok else "SKIPPED"
    STATUS.append({"source": source, "status": label, "records": n, "note": note, "run_at": now_utc()})
    print(f"[{label}] {source}: {n} records. {note}")


def request_json(method, url, retries=1, **kwargs):
    """GET or POST with one retry on a timeout or server error. Raises on anything else."""
    for attempt in range(retries + 1):
        try:
            r = requests.request(method, url, headers=HEADERS, timeout=TIMEOUT, **kwargs)
            if r.status_code >= 500 and attempt < retries:
                time.sleep(2)
                continue
            r.raise_for_status()
            return r.json()
        except requests.Timeout:
            if attempt < retries:
                time.sleep(2)
                continue
            raise

In [3]:
def fetch_federal_register():
    """Rules, proposed rules, notices, and presidential documents from DoD in the signal window."""
    url = "https://www.federalregister.gov/api/v1/documents.json"
    params = {
        "conditions[term]": FR_TERM,
        "conditions[agencies][]": FR_AGENCY_SLUGS,
        "conditions[publication_date][gte]": START.isoformat(),
        "per_page": 1000,
        "order": "newest",
        "fields[]": ["document_number", "title", "type", "abstract",
                     "publication_date", "html_url", "agencies"],
    }
    pages = [request_json("GET", url, params=params)]
    while pages[-1].get("next_page_url") and len(pages) < MAX_PAGES:
        pages.append(request_json("GET", pages[-1]["next_page_url"]))

    path = save_raw("federal_register", {"retrieved_at": now_utc(), "url": url,
                                         "params": params, "pages": pages})
    records = [doc for page in pages for doc in page.get("results", [])]
    return records, f"{LOOKBACK_DAYS}-day window. Raw saved to {path}"


def fetch_usaspending():
    """DoD contract awards (types A-D) matching the award keywords in the award window."""
    url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
    body = {
        "filters": {
            "keywords": AWARD_KEYWORDS,
            "award_type_codes": ["A", "B", "C", "D"],
            "time_period": [{"start_date": AWARD_START.isoformat(), "end_date": TODAY.isoformat()}],
            "agencies": [{"type": "awarding", "tier": "toptier", "name": AGENCY_NAME}],
        },
        "fields": ["Award ID", "Recipient Name", "Start Date", "End Date", "Award Amount",
                   "Description", "Awarding Agency", "Awarding Sub Agency",
                   "Contract Award Type", "Last Modified Date", "generated_internal_id"],
        "page": 1, "limit": 100, "sort": "Award Amount", "order": "desc", "subawards": False,
    }
    pages = [request_json("POST", url, json=body)]
    while pages[-1].get("page_metadata", {}).get("hasNext") and len(pages) < MAX_PAGES:
        body = {**body, "page": body["page"] + 1}
        pages.append(request_json("POST", url, json=body))

    path = save_raw("usaspending", {"retrieved_at": now_utc(), "url": url,
                                    "body": body, "pages": pages})
    records = [award for page in pages for award in page.get("results", [])]
    return records, f"{AWARD_LOOKBACK_DAYS}-day window (DoD 90-day reporting delay). Raw saved to {path}"


def fetch_sam():
    """SAM.gov opportunities with the focus term in the title, kept to DoD after the raw save."""
    if not SAM_API_KEY:
        raise RuntimeError("SAM_API_KEY is not set in this environment")

    cache = RAW_DIR / f"sam_{TODAY:%Y%m%d}.json"
    if cache.exists() and not SAM_REFRESH:
        payload = json.loads(cache.read_text(encoding="utf-8"))
        source_note = f"Reused today's pull from {cache} (daily request cap)"
    else:
        url = "https://api.sam.gov/opportunities/v2/search"
        params = {
            "api_key": SAM_API_KEY,
            "postedFrom": START.strftime("%m/%d/%Y"),   # SAM.gov wants MM/dd/yyyy
            "postedTo": TODAY.strftime("%m/%d/%Y"),
            "title": SAM_TITLE,
            "limit": 1000,
            "offset": 0,
        }
        response = request_json("GET", url, params=params, retries=0)  # no retry: each call counts
        safe_params = {k: v for k, v in params.items() if k != "api_key"}
        payload = {"retrieved_at": now_utc(), "url": url, "params": safe_params, "response": response}
        save_raw("sam", payload)
        source_note = f"{LOOKBACK_DAYS}-day window. Raw saved to {cache}"

    response = payload["response"]
    all_opps = response.get("opportunitiesData") or []
    if response.get("totalRecords", 0) > len(all_opps):
        source_note += f". Warning: {response['totalRecords']} matched, only {len(all_opps)} returned"
    dod = [o for o in all_opps
           if str(o.get("fullParentPathName") or "").upper().startswith(SAM_DEPT_PREFIX)]
    return dod, f"{len(dod)} of {len(all_opps)} title matches are DoD. {source_note}"

In [4]:
SOURCES = [
    ("federal_register", fetch_federal_register),
    ("usaspending", fetch_usaspending),
    ("sam", fetch_sam),
]

RAW_RECORDS = {}
for name, fetch in SOURCES:
    try:
        records, note = fetch()
        RAW_RECORDS[name] = records
        log_status(name, True, len(records), note)
    except Exception as err:                       # keep going; one bad source should not stop the run
        RAW_RECORDS[name] = []
        log_status(name, False, 0, redact(f"{type(err).__name__}: {err}. Continuing without it."))

status_df = pd.DataFrame(STATUS)
status_df

[OK] federal_register: 5 records. 90-day window. Raw saved to data\raw\federal_register_20261011.json


[OK] usaspending: 30 records. 365-day window (DoD 90-day reporting delay). Raw saved to data\raw\usaspending_20261011.json


[OK] sam: 0 records. 0 of 2 title matches are DoD. 90-day window. Raw saved to data\raw\sam_20261011.json


,source,status,records,note,run_at
0,federal_register,OK,5,90-day window. Raw saved to data\raw\federal_register_20261011.json,2026-10-12T01:23:08+00:00
1,usaspending,OK,30,365-day window (DoD 90-day reporting delay). Raw saved to data\raw\usaspending_20261011.json,2026-10-12T01:23:09+00:00
2,sam,OK,0,0 of 2 title matches are DoD. 90-day window. Raw saved to data\raw\sam_20261011.json,2026-10-12T01:23:10+00:00


## 2. Normalize and merge

Each source maps into one schema: `source`, `id`, `title`, `agency`, `date`, `doc_type`, `url`, `text`, `amount`, `retrieved_at`.

Two rules matter here. A record with no date keeps a blank date, because filling in today's date would make an old signal look new. Duplicates are dropped on `source` plus `id`. Assert checks run after the merge, so a bad join stops the notebook instead of flowing into the tiers.

In [5]:
SCHEMA = ["source", "id", "title", "agency", "date", "doc_type", "url", "text", "amount", "retrieved_at"]


def clean(value):
    """Strip strings and treat empty or whitespace-only values as missing."""
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value or None
    return value


def raw_retrieved_at(source):
    """Pull time recorded in the raw file, so a reused SAM.gov cache keeps its original time."""
    path = RAW_DIR / f"{source}_{TODAY:%Y%m%d}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8")).get("retrieved_at")


def from_federal_register(doc, retrieved_at):
    agencies = [clean(a.get("name")) or clean(a.get("raw_name")) for a in (doc.get("agencies") or [])]
    return {
        "source": "federal_register",
        "id": clean(doc.get("document_number")),
        "title": clean(doc.get("title")),
        "agency": "; ".join(a for a in agencies if a) or None,
        "date": clean(doc.get("publication_date")),
        "doc_type": clean(doc.get("type")),
        "url": clean(doc.get("html_url")),
        "text": clean(doc.get("abstract")),
        "amount": None,
        "retrieved_at": retrieved_at,
    }


def from_usaspending(award, retrieved_at):
    internal_id = clean(award.get("generated_internal_id"))
    top, sub = clean(award.get("Awarding Agency")), clean(award.get("Awarding Sub Agency"))
    award_id = clean(award.get("Award ID")) or "unknown ID"
    recipient = clean(award.get("Recipient Name")) or "unknown recipient"
    return {
        "source": "usaspending",
        "id": internal_id,
        "title": f"Award {award_id} to {recipient}",
        "agency": f"{top} ({sub})" if top and sub else (top or sub),
        "date": clean(award.get("Start Date")),
        "doc_type": clean(award.get("Contract Award Type")),
        "url": f"https://www.usaspending.gov/award/{internal_id}" if internal_id else None,
        "text": clean(award.get("Description")),
        "amount": award.get("Award Amount"),
        "retrieved_at": retrieved_at,
    }


def from_sam(opp, retrieved_at):
    notice_id = clean(opp.get("noticeId"))
    org_path = clean(opp.get("fullParentPathName"))
    # The full SAM.gov description is a separate call that counts against the daily cap, so the
    # text field carries the structured details instead.
    details = [
        f"Solicitation {clean(opp.get('solicitationNumber'))}" if clean(opp.get("solicitationNumber")) else None,
        f"NAICS {clean(opp.get('naicsCode'))}" if clean(opp.get("naicsCode")) else None,
        f"Responses due {clean(opp.get('responseDeadLine'))}" if clean(opp.get("responseDeadLine")) else None,
    ]
    return {
        "source": "sam",
        "id": notice_id,
        "title": clean(opp.get("title")),
        "agency": org_path.replace(".", " > ") if org_path else None,
        "date": clean(opp.get("postedDate")),
        "doc_type": clean(opp.get("type")),
        "url": clean(opp.get("uiLink")) or (f"https://sam.gov/opp/{notice_id}/view" if notice_id else None),
        "text": ". ".join(d for d in details if d) or None,
        "amount": None,
        "retrieved_at": retrieved_at,
    }


MAPPERS = {"federal_register": from_federal_register, "usaspending": from_usaspending, "sam": from_sam}

In [6]:
rows = []
for source, records in RAW_RECORDS.items():
    retrieved = raw_retrieved_at(source)
    rows.extend(MAPPERS[source](record, retrieved) for record in records)

signals = pd.DataFrame(rows, columns=SCHEMA)

# Dates: parse what exists and leave missing as missing (NaT). Never default to today.
# Every source starts its date with YYYY-MM-DD, so parse the first 10 characters.
signals["_raw_date_missing"] = signals["date"].isna()
signals["date"] = pd.to_datetime(signals["date"].astype("string").str[:10], format="%Y-%m-%d", errors="coerce")
signals["amount"] = pd.to_numeric(signals["amount"], errors="coerce")

# Duplicates: same source + same ID is the same signal.
before = len(signals)
signals = signals.drop_duplicates(subset=["source", "id"], keep="first")
dupes_dropped = before - len(signals)

signals = signals.sort_values("date", ascending=False, na_position="last").reset_index(drop=True)

# ---- Checks after the merge ----
missing_ids = signals.loc[signals["id"].isna(), ["source", "title"]]
assert missing_ids.empty, f"Records with no ID:\n{missing_ids.to_string()}"

dupe_keys = signals.loc[signals.duplicated(["source", "id"], keep=False), ["source", "id"]]
assert dupe_keys.empty, f"Duplicate source + ID keys:\n{dupe_keys.to_string()}"

bad_urls = signals.loc[~signals["url"].fillna("").str.startswith("https://"), ["source", "id", "url"]]
assert bad_urls.empty, f"URLs missing or not https:\n{bad_urls.to_string()}"

filled = signals.loc[signals["_raw_date_missing"] & signals["date"].notna(), ["source", "id"]]
assert filled.empty, f"A missing date was filled in:\n{filled.to_string()}"

# ---- Summary ----
blank_dates = int(signals["date"].isna().sum())
unreadable = int((signals["date"].isna() & ~signals["_raw_date_missing"]).sum())
signals = signals.drop(columns="_raw_date_missing")

if signals.empty:
    print("No records to merge. Check the status lines in section 1.")
print(f"{len(signals)} signals after merge ({dupes_dropped} duplicates dropped)")
print(signals["source"].value_counts().to_string())
print(f"{blank_dates} records have no date and stay blank ({unreadable} had a date that could not be read)")
print("Checks passed: every record has an ID, no duplicate keys, every URL is https, no missing date was filled in")

signals[["source", "date", "doc_type", "title", "agency"]].head(15)

35 signals after merge (0 duplicates dropped)
source
usaspending         30
federal_register     5
0 records have no date and stay blank (0 had a date that could not be read)
Checks passed: every record has an ID, no duplicate keys, every URL is https, no missing date was filled in


,source,date,doc_type,title,agency
0,federal_register,2026-09-25,Notice,Privacy Act of 1974; System of Records,Defense Department; Navy Department
1,federal_register,2026-09-18,Proposed Rule,"Federal Acquisition Regulation: Revolutionary FAR Overhaul Parts 8, 12, 13, 15, 38, 44, and 51",Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration
2,federal_register,2026-09-18,Proposed Rule,"Federal Acquisition Regulation: Revolutionary Federal Acquisition Regulation Overhaul Parts 14, 28, 36, and 52",Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration
3,federal_register,2026-09-11,Notice,Establishment of Department of Defense Federal Advisory Committees-Homeland Defense Board,Defense Department; Defense Department
4,federal_register,2026-08-14,Notice,Regulatory Agenda,Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration
5,usaspending,2026-07-09,DELIVERY ORDER,Award N6339426F4021 to COLOSSAL CONTRACTING LLC,Department of Defense (Department of the Navy)
6,usaspending,2026-07-06,DEFINITIVE CONTRACT,Award W912HQ26CA041 to EATON CORPORATION,Department of Defense (Department of the Army)
7,usaspending,2026-07-01,DEFINITIVE CONTRACT,Award HTC71126CE129 to BOOZ ALLEN HAMILTON INC,Department of Defense (USTRANSCOM)
8,usaspending,2026-07-01,PURCHASE ORDER,Award FA864926P0246 to SNO CORP.,Department of Defense (Department of the Air Force)
9,usaspending,2026-06-30,PURCHASE ORDER,Award FA864926P0140 to VALIDVOICE LLC,Department of Defense (Department of the Air Force)


## 3. Evidence tiers

Plain rules, no machine learning and no LLM. Rules are checked in order and the first match wins. Anything no rule claims falls to Tier 3.

| Tier | What lands here |
|---|---|
| 1 | Federal Register final rules, proposed rules, and presidential documents |
| 2 | Federal Register notices, contract awards, and SAM.gov notices |
| 3 | Everything else |

Every row gets a reason string naming the rule that placed it. The tier measures how strong the type of evidence is, not whether the record is on topic: a privacy notice that mentions cybersecurity still lands in Tier 2 by rule.

In [7]:
# Edit this list to change the ladder. Order matters: the first matching rule wins.
TIER_RULES = [
    {"rule": "T1-RULE",      "tier": 1, "source": "federal_register", "doc_types": {"Rule"},
     "reason": "Federal Register final rule"},
    {"rule": "T1-PRORULE",   "tier": 1, "source": "federal_register", "doc_types": {"Proposed Rule"},
     "reason": "Federal Register proposed rule"},
    {"rule": "T1-PRESDOC",   "tier": 1, "source": "federal_register", "doc_types": {"Presidential Document"},
     "reason": "Presidential document"},
    {"rule": "T2-FR-NOTICE", "tier": 2, "source": "federal_register", "doc_types": {"Notice"},
     "reason": "Federal Register notice"},
    {"rule": "T2-AWARD",     "tier": 2, "source": "usaspending",      "doc_types": None,
     "reason": "Contract award on USAspending"},
    {"rule": "T2-SAM",       "tier": 2, "source": "sam",              "doc_types": None,
     "reason": "SAM.gov notice"},
]
FALLBACK_RULE = "T3-OTHER"


def assign_tier(record):
    """Return (tier, rule ID, reason) for one signal. doc_types=None means any type from that source."""
    doc_type = record["doc_type"] if isinstance(record["doc_type"], str) else ""
    for rule in TIER_RULES:
        if record["source"] != rule["source"]:
            continue
        if rule["doc_types"] and doc_type.lower() not in {t.lower() for t in rule["doc_types"]}:
            continue
        detail = f" ({doc_type})" if rule["doc_types"] is None and doc_type else ""
        return rule["tier"], rule["rule"], f"Tier {rule['tier']}, rule {rule['rule']}: {rule['reason']}{detail}"
    return 3, FALLBACK_RULE, (f"Tier 3, rule {FALLBACK_RULE}: no Tier 1 or 2 rule covers "
                              f"{record['source']} / {doc_type or 'no document type'}")


results = [assign_tier(r) for r in signals.to_dict("records")]
signals["tier"] = [tier for tier, _, _ in results]
signals["tier_rule"] = [rule for _, rule, _ in results]
signals["tier_reason"] = [reason for _, _, reason in results]

# ---- Checks ----
assert signals["tier"].isin([1, 2, 3]).all(), "A row has a tier outside 1-3"
assert all(isinstance(r, str) and r for r in signals["tier_reason"]), "A row has no tier reason"

# ---- Ranked view: strongest evidence first, newest first within a tier, blank dates last ----
ranked = signals.sort_values(["tier", "date"], ascending=[True, False], na_position="last").reset_index(drop=True)

print("Rows per rule:")
print(signals.groupby(["tier", "tier_rule"]).size().to_string())
ranked[["tier", "date", "source", "title", "tier_reason"]].head(20)

Rows per rule:
tier  tier_rule   
1     T1-PRORULE       2
2     T2-AWARD        30
      T2-FR-NOTICE     3


,tier,date,source,title,tier_reason
0,1,2026-09-18,federal_register,"Federal Acquisition Regulation: Revolutionary FAR Overhaul Parts 8, 12, 13, 15, 38, 44, and 51","Tier 1, rule T1-PRORULE: Federal Register proposed rule"
1,1,2026-09-18,federal_register,"Federal Acquisition Regulation: Revolutionary Federal Acquisition Regulation Overhaul Parts 14, 28, 36, and 52","Tier 1, rule T1-PRORULE: Federal Register proposed rule"
2,2,2026-09-25,federal_register,Privacy Act of 1974; System of Records,"Tier 2, rule T2-FR-NOTICE: Federal Register notice"
3,2,2026-09-11,federal_register,Establishment of Department of Defense Federal Advisory Committees-Homeland Defense Board,"Tier 2, rule T2-FR-NOTICE: Federal Register notice"
4,2,2026-08-14,federal_register,Regulatory Agenda,"Tier 2, rule T2-FR-NOTICE: Federal Register notice"
5,2,2026-07-09,usaspending,Award N6339426F4021 to COLOSSAL CONTRACTING LLC,"Tier 2, rule T2-AWARD: Contract award on USAspending (DELIVERY ORDER)"
6,2,2026-07-06,usaspending,Award W912HQ26CA041 to EATON CORPORATION,"Tier 2, rule T2-AWARD: Contract award on USAspending (DEFINITIVE CONTRACT)"
7,2,2026-07-01,usaspending,Award HTC71126CE129 to BOOZ ALLEN HAMILTON INC,"Tier 2, rule T2-AWARD: Contract award on USAspending (DEFINITIVE CONTRACT)"
8,2,2026-07-01,usaspending,Award FA864926P0246 to SNO CORP.,"Tier 2, rule T2-AWARD: Contract award on USAspending (PURCHASE ORDER)"
9,2,2026-06-30,usaspending,Award FA864926P0140 to VALIDVOICE LLC,"Tier 2, rule T2-AWARD: Contract award on USAspending (PURCHASE ORDER)"


## 4. Mockup: signal feed (S1) and signal detail (S2)

Renders the ranked table from section 3 as the two Module 4 screens, using this run's real signals. The feed shows tier, title, tier reason, agency, evidence source, and date. The detail view quotes the record's own text, names the rule that placed it, and links back to the source.

Two features from the wireframes are deliberately left as labeled placeholders: the lead-time estimate and the LLM-drafted rationale. The full mockup, with a detail card for every signal, is saved to `mockup/radar_mockup.html` to open in a browser.

In [8]:
import html
from IPython.display import HTML, display

MOCKUP_DIR = Path("mockup")
MOCKUP_DIR.mkdir(exist_ok=True)
SOURCE_LABELS = {"federal_register": "Federal Register", "usaspending": "USAspending", "sam": "SAM.gov"}
QUOTE_LIMIT = 600   # characters of source text quoted on the detail card

CSS = """<style>
.radar{font-family:"Segoe UI",-apple-system,Roboto,Helvetica,Arial,sans-serif;color:#1d2733;background:#fff;max-width:1440px;font-size:14px;line-height:1.45}
.radar *{box-sizing:border-box}
.radar .bar{display:flex;align-items:center;gap:28px;padding:14px 20px;border-bottom:1px solid #d9dee5}
.radar .logo{font-weight:700;font-size:18px;display:flex;align-items:center;gap:10px}
.radar .logo i{width:20px;height:20px;border-radius:50%;border:6px solid #1f3a5f;display:inline-block}
.radar .nav span{margin-right:22px;color:#4b5563}
.radar .nav .on{font-weight:600;color:#111;background:#eef1f5;padding:6px 10px;border-radius:6px}
.radar .proto{margin-left:auto;color:#6b7280;font-size:12px}
.radar .wrap{padding:18px 20px}
.radar .summary{color:#4b5563;margin-bottom:10px}
.radar table{width:100%;border-collapse:collapse;border:1px solid #d9dee5}
.radar th{background:#f3f5f8;text-align:left;font-weight:600;padding:10px 12px;border-bottom:1px solid #d9dee5;font-size:13px}
.radar td{padding:12px;border-bottom:1px solid #e5e9ef;vertical-align:top}
.radar .badge{display:inline-block;padding:4px 12px;border-radius:12px;font-weight:600;font-size:12px;white-space:nowrap}
.radar .t1{background:#1f3a5f;color:#fff}.radar .t2{background:#4a6585;color:#fff}.radar .t3{background:#b7c6d8;color:#1d2733}
.radar .title{font-weight:600;font-size:15px}
.radar .reason{color:#4b5563;font-size:12px;margin-top:3px}
.radar a{color:#2563a8;text-decoration:none}
.radar .chip{display:inline-block;border:1px solid #cfd6df;border-radius:12px;padding:2px 10px;font-size:12px;background:#f7f8fa}
.radar .muted{color:#6b7280;font-size:12px}
.radar .none{color:#6b7280;font-style:italic}
.radar .ph{border:1px dashed #9fb0c4;border-radius:6px;padding:8px 10px;color:#4b5563;font-size:12px;background:#fafbfc}
.radar .ph b{color:#1d2733}
.radar .status{margin-top:14px;padding:10px 14px;border:1px solid #d9dee5;border-radius:8px;background:#f5f7fa;font-size:13px}
.radar .dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin:0 6px 0 14px}
.radar .ok{background:#2e9e5b}.radar .bad{background:#e0663a}
.radar .card{border:1px solid #d9dee5;border-radius:8px;padding:16px 18px;margin-top:14px;background:#fff}
.radar .detail{border-top:3px solid #1f3a5f;margin-top:28px;padding-top:6px}
.radar .quote{border:1px solid #e1e6ec;background:#f8f9fb;border-radius:6px;padding:12px 14px;margin:10px 0}
.radar .quote q{font-style:italic}
.radar h2{font-size:22px;margin:4px 0 2px;display:inline;vertical-align:middle}
.radar h3{font-size:16px;margin:0 0 6px}
.radar .grid{display:grid;grid-template-columns:2fr 1fr;gap:16px}
.radar .foot{margin-top:14px;padding:10px 14px;border:1px solid #d9dee5;border-radius:8px;background:#f5f7fa;color:#4b5563;font-size:12.5px}
</style>"""

esc = lambda value: html.escape(str(value)) if value is not None and not (isinstance(value, float) and pd.isna(value)) else ""


def fmt_date(value):
    return '<span class="none">no date</span>' if pd.isna(value) else f"{value:%b} {value.day}, {value.year}"


def fmt_ts(value):
    """ISO timestamp from the pull, shown as e.g. Oct 12, 00:50 UTC."""
    try:
        ts = pd.Timestamp(value)
        return f"{ts:%b} {ts.day}, {ts:%H:%M} UTC"
    except (ValueError, TypeError):
        return esc(value)


def fmt_amount(value):
    if pd.isna(value):
        return ""
    return f"${value / 1e6:,.1f}M" if value >= 1e6 else f"${value:,.0f}"


def badge(tier):
    return f'<span class="badge t{int(tier)}">Tier {int(tier)}</span>'


def header(active):
    tabs = "".join(f'<span class="{"on" if t == active else ""}">{t}</span>'
                   for t in ["Feed", "Watchlist", "Digest", "Sources", "Settings"])
    return (f'<div class="bar"><div class="logo"><i></i>Policy to Pipeline Radar</div>'
            f'<div class="nav">{tabs}</div>'
            f'<div class="proto">Prototype · live public data pulled {TODAY:%b %d, %Y}</div></div>')


def status_line():
    parts = []
    for s in STATUS:
        label = SOURCE_LABELS.get(s["source"], s["source"])
        if s["status"] == "OK":
            parts.append(f'<span class="dot ok"></span>{label}: {s["records"]} records')
        else:
            parts.append(f'<span class="dot bad"></span>{label} skipped: {esc(s["note"][:110])}')
    run_at = STATUS[0]["run_at"] if STATUS else "not run"
    return f'<div class="status">Last pull {fmt_ts(run_at)}{"".join(parts)}</div>'


def feed_html(df):
    if df.empty:
        return '<div class="card none">No signals in this run. Check the status line below.</div>'
    rows = []
    for i, r in df.iterrows():
        amount = fmt_amount(r["amount"])
        rows.append(
            f'<tr><td>{badge(r["tier"])}</td>'
            f'<td><div class="title">{esc(r["title"])}</div>'
            f'<div class="reason">{esc(r["tier_reason"])}</div>'
            f'<a href="#detail-{i}">why this tier &rsaquo;</a></td>'
            f'<td>{esc(r["agency"]) or "<span class=none>no agency</span>"}</td>'
            f'<td><span class="chip">{SOURCE_LABELS[r["source"]]}</span>'
            f'<div class="muted">{esc(r["doc_type"])}{" · " + amount if amount else ""}</div></td>'
            f'<td style="white-space:nowrap">{fmt_date(r["date"])}</td>'
            f'<td><div class="ph"><b>Placeholder</b><br>Lead-time estimate not built</div></td></tr>')
    head = ("<tr><th>Tier</th><th>Signal and tier reason</th><th>Agency</th>"
            "<th>Evidence (public source)</th><th>Date</th><th>Est. lead time</th></tr>")
    return f"<table>{head}{''.join(rows)}</table>"


def quote_text(text):
    if not isinstance(text, str) or not text:
        return '<span class="none">The source record has no text to quote. Open the document to read it.</span>'
    if len(text) > QUOTE_LIMIT:
        text = text[:QUOTE_LIMIT].rsplit(" ", 1)[0] + " [...]"
    return f"<q>{esc(text)}</q>"


def detail_html(r, i):
    source = SOURCE_LABELS[r["source"]]
    amount = fmt_amount(r["amount"])
    meta = " · ".join(x for x in [esc(r["agency"]), source, esc(r["doc_type"]), amount,
                                   fmt_date(r["date"]), f'retrieved {fmt_ts(r["retrieved_at"])}'] if x)
    reason = esc(r["tier_reason"]).split(": ", 1)[-1]
    return f"""
<div class="detail" id="detail-{i}">
  <div class="muted">Feed &rsaquo; {esc(r["title"])}</div>
  <div style="margin:6px 0">{badge(r["tier"])} <h2>{esc(r["title"])}</h2></div>
  <div class="muted">{meta}</div>
  <div class="grid">
    <div class="card">
      <h3>Why this is Tier {int(r["tier"])}</h3>
      <div>Rule <b>{esc(r["tier_rule"])}</b> placed it here: {reason}.</div>
      <div class="quote"><b>{source} · {esc(r["doc_type"])} · {fmt_date(r["date"])}</b><br>{quote_text(r["text"])}<br>
        <a href="{esc(r["url"])}" target="_blank">Open source document &#8599;</a></div>
      <div class="ph"><b>Placeholder: LLM rationale.</b> A later version drafts a short rationale from the quoted
        passage only, with a "does this match the source?" check. Not built in this prototype.</div>
    </div>
    <div class="card">
      <h3>Estimated lead time</h3>
      <div class="ph"><b>Placeholder: lead-time estimate.</b> Not built in this prototype. It needs pilot history
        before it can show an honest range.</div>
    </div>
  </div>
  <div class="foot">No model output on this page. The text is quoted from the source record and the tier comes from a written rule.</div>
</div>"""


def summary_line(df):
    counts = df["tier"].value_counts().sort_index()
    tiers = " · ".join(f"Tier {int(t)}: {n}" for t, n in counts.items()) or "no signals"
    return f'<div class="summary">{len(df)} signals · {tiers} · strongest evidence first, newest first within a tier</div>'


feed_section = summary_line(ranked) + feed_html(ranked) + status_line()
details_section = "".join(detail_html(r, i) for i, r in ranked.iterrows())

page = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Policy to Pipeline Radar: mockup</title>{CSS}</head>
<body style="margin:0;background:#fff"><div class="radar">{header("Feed")}<div class="wrap">
{feed_section}<h3 style="margin-top:28px">Signal detail (S2), one card per signal</h3>{details_section}
</div></div></body></html>"""

mockup_path = MOCKUP_DIR / "radar_mockup.html"
mockup_path.write_text(page, encoding="utf-8")
print(f"Saved {mockup_path} ({len(ranked)} signals). Open it in a browser for the full feed and every detail card.")

Saved mockup\radar_mockup.html (35 signals). Open it in a browser for the full feed and every detail card.


In [9]:
# S1: signal feed
display(HTML(CSS + f'<div class="radar">{header("Feed")}<div class="wrap">{feed_section}</div></div>'))

Tier,Signal and tier reason,Agency,Evidence (public source),Date,Est. lead time
Tier 1,"Federal Acquisition Regulation: Revolutionary FAR Overhaul Parts 8, 12, 13, 15, 38, 44, and 51Tier 1, rule T1-PRORULE: Federal Register proposed rulewhy this tier ›",Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration,Federal RegisterProposed Rule,"Sep 18, 2026",PlaceholderLead-time estimate not built
Tier 1,"Federal Acquisition Regulation: Revolutionary Federal Acquisition Regulation Overhaul Parts 14, 28, 36, and 52Tier 1, rule T1-PRORULE: Federal Register proposed rulewhy this tier ›",Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration,Federal RegisterProposed Rule,"Sep 18, 2026",PlaceholderLead-time estimate not built
Tier 2,"Privacy Act of 1974; System of RecordsTier 2, rule T2-FR-NOTICE: Federal Register noticewhy this tier ›",Defense Department; Navy Department,Federal RegisterNotice,"Sep 25, 2026",PlaceholderLead-time estimate not built
Tier 2,"Establishment of Department of Defense Federal Advisory Committees-Homeland Defense BoardTier 2, rule T2-FR-NOTICE: Federal Register noticewhy this tier ›",Defense Department; Defense Department,Federal RegisterNotice,"Sep 11, 2026",PlaceholderLead-time estimate not built
Tier 2,"Regulatory AgendaTier 2, rule T2-FR-NOTICE: Federal Register noticewhy this tier ›",Management and Budget Office; Federal Procurement Policy Office; Defense Department; General Services Administration; National Aeronautics and Space Administration,Federal RegisterNotice,"Aug 14, 2026",PlaceholderLead-time estimate not built
Tier 2,"Award N6339426F4021 to COLOSSAL CONTRACTING LLCTier 2, rule T2-AWARD: Contract award on USAspending (DELIVERY ORDER)why this tier ›",Department of Defense (Department of the Navy),"USAspendingDELIVERY ORDER · $81,306","Jul 9, 2026",PlaceholderLead-time estimate not built
Tier 2,"Award W912HQ26CA041 to EATON CORPORATIONTier 2, rule T2-AWARD: Contract award on USAspending (DEFINITIVE CONTRACT)why this tier ›",Department of Defense (Department of the Army),"USAspendingDEFINITIVE CONTRACT · $263,715","Jul 6, 2026",PlaceholderLead-time estimate not built
Tier 2,"Award HTC71126CE129 to BOOZ ALLEN HAMILTON INCTier 2, rule T2-AWARD: Contract award on USAspending (DEFINITIVE CONTRACT)why this tier ›",Department of Defense (USTRANSCOM),USAspendingDEFINITIVE CONTRACT · $13.7M,"Jul 1, 2026",PlaceholderLead-time estimate not built
Tier 2,"Award FA864926P0246 to SNO CORP.Tier 2, rule T2-AWARD: Contract award on USAspending (PURCHASE ORDER)why this tier ›",Department of Defense (Department of the Air Force),"USAspendingPURCHASE ORDER · $110,000","Jul 1, 2026",PlaceholderLead-time estimate not built
Tier 2,"Award FA864926P0140 to VALIDVOICE LLCTier 2, rule T2-AWARD: Contract award on USAspending (PURCHASE ORDER)why this tier ›",Department of Defense (Department of the Air Force),"USAspendingPURCHASE ORDER · $74,627","Jun 30, 2026",PlaceholderLead-time estimate not built


In [10]:
# S2: signal detail. Change DETAIL_ROW to view another signal (row number in the ranked table).
DETAIL_ROW = 0
if ranked.empty:
    print("No signals to show.")
else:
    display(HTML(CSS + f'<div class="radar">{header("Feed")}<div class="wrap">'
                 f'{detail_html(ranked.loc[DETAIL_ROW], DETAIL_ROW)}</div></div>'))

## 5. Ask the Radar (S3), rule-based stand-in for the LLM version

In the Module 4 design, a language model turns a plain question into a structured filter and shows it as chips before anything runs. This prototype has no LLM. A fixed set of rules does the same job here, and every step after the chips works exactly as the LLM version would.

Three steps, in three cells:
1. **Read the question** into chips for tier, source, agency, date range, and keyword. Any leap the parser makes is spelled out, and the words it could not use are listed.
2. **Check the chips.** Nothing has run yet. To fix a misreading, edit `filters` directly in the next cell before running it.
3. **Run the filter** on this run's signals and list the matching records. Results are rows from the feed, never generated text.

In [11]:
import re

ASK_LABEL = "Rule-based stand-in for the LLM version"

# What a user might type -> text to look for in the agency field (case-insensitive)
AGENCY_ALIASES = {
    "air force": "AIR FORCE", "army": "ARMY", "navy": "NAVY", "marine corps": "MARINE",
    "space force": "SPACE FORCE", "disa": "DEFENSE INFORMATION SYSTEMS", "dla": "DEFENSE LOGISTICS",
    "dcsa": "COUNTERINTELLIGENCE AND SECURITY", "whs": "WASHINGTON HEADQUARTERS",
    "transcom": "TRANSCOM", "darpa": "ADVANCED RESEARCH PROJECTS",
}
DOD_WORDS = r"\b(dod|department of defense|defense department|pentagon)\b"

# Words -> source. The first group in each pattern is what gets quoted back in the note.
SOURCE_PATTERNS = [
    (r"\b(federal register|proposed rules?|rules?|regulations?|rulemaking)\b", "federal_register"),
    (r"\b(usaspending|awards?|contracts?|spending)\b", "usaspending"),
    (r"\b(sam\.gov|sam|opportunit(?:y|ies)|solicitations?|sources sought|presolicitations?|rfis?)\b", "sam"),
]
TOPIC_WORDS = ["zero trust", "cmmc", "cybersecurity", "cyber", "icam", "identity", "cloud", "privacy",
               "artificial intelligence", "endpoint", "network"]
STOPWORDS = set("""a an and any anything are all at by can do does for from give have i in is it list me
    my of on or over please show signals signal tell that the there these this those to us what which with
    within get find ones s has have been was were be""".split())
UNIT_DAYS = {"day": 1, "week": 7, "month": 30, "year": 365}


def parse_question(question):
    """Turn a plain question into a filter dict, notes on any leaps, and the words not used."""
    q = question.lower()
    used = []                                   # character spans the rules consumed
    filters = {"tier": [], "source": [], "agency": [], "window": None, "keyword": []}
    notes = []

    def take(match):
        used.append(match.span())

    # Tier: "tier 1", "tiers 1 and 2", "tier 1-2"; "strongest" is a leap to Tier 1
    m = re.search(r"\btiers?\s*((?:[1-3]\s*(?:,|and|or|&|/|-|to|through)?\s*)+)", q)
    if m:
        take(m)
        nums = [int(n) for n in re.findall(r"[1-3]", m.group(1))]
        if re.search(r"[1-3]\s*(?:-|to|through)\s*[1-3]", m.group(1)):
            nums = list(range(min(nums), max(nums) + 1))
        filters["tier"] = sorted(set(nums))
    else:
        m = re.search(r"\b(strongest|top tier|best evidence|strong evidence)\b", q)
        if m:
            take(m)
            filters["tier"] = [1]
            notes.append(f'"{m.group(1)}" became Tier 1, the strongest rung of the evidence ladder.')

    # Source
    for pattern, source in SOURCE_PATTERNS:
        for m in re.finditer(pattern, q):
            take(m)
            if source not in filters["source"]:
                filters["source"].append(source)
                if m.group(1) not in ("federal register", "usaspending", "sam.gov", "sam"):
                    notes.append(f'"{m.group(1)}" became Source: {SOURCE_LABELS[source]}.')

    # Agency
    for alias, needle in AGENCY_ALIASES.items():
        m = re.search(rf"\b{re.escape(alias)}\b", q)
        if m:
            take(m)
            filters["agency"].append((alias.upper() if len(alias) <= 5 else alias.title(), needle))
    m = re.search(DOD_WORDS, q)
    if m:
        take(m)
        notes.append(f'"{m.group(1)}" added no filter: every signal in this run is already DoD.')

    # Date range
    m = re.search(r"\b(?:last|past|previous)\s+(\d+)\s+(day|week|month|year)s?\b", q)
    m1 = re.search(r"\b(?:last|past)\s+(day|week|month|year)\b", q)
    if m:
        take(m)
        days, label = int(m.group(1)) * UNIT_DAYS[m.group(2)], f"last {m.group(1)} {m.group(2)}s"
    elif m1:
        take(m1)
        days, label = UNIT_DAYS[m1.group(1)], f"last {m1.group(1)}"
    elif (m := re.search(r"\bthis (month|year)\b", q)):
        take(m)
        start = TODAY.replace(day=1) if m.group(1) == "month" else TODAY.replace(month=1, day=1)
        days, label = (TODAY - start).days, f"this {m.group(1)}"
    elif (m := re.search(r"\b(new|recent|latest|lately)\b", q)):
        take(m)
        days, label = 30, "last 30 days"
        notes.append(f'"{m.group(1)}" became the last 30 days.')
    else:
        days = None
    if days is not None and (m := re.search(r"\b(new|recent|latest|lately)\b", q)):
        take(m)                                 # "new" is covered by the explicit window
    if days is not None:
        start = TODAY - timedelta(days=days)
        filters["window"] = {"start": start,
                             "label": f"{label} ({start:%b} {start.day}, {start.year} to {TODAY:%b} {TODAY.day}, {TODAY.year})"}
        awards_possible = (not filters["source"] or "usaspending" in filters["source"]) and \
            (not filters["tier"] or 2 in filters["tier"])
        if days < 90 and awards_possible:
            notes.append("DoD awards post on a 90-day delay, so a window this short will miss recent awards.")

    # Keyword: quoted phrases, "about X", or known topic words
    for m in re.finditer(r'["“]([^"”]+)["”]', q):
        take(m)
        filters["keyword"].append(m.group(1).strip())
    m = re.search(r"\b(?:about|mentioning|regarding|related to)\s+([a-z0-9][a-z0-9 \-]{1,40}?)"
                  r"(?=\s+(?:from|in|over|since|during|this|last|past|at|by)\b|[?.!,]|$)", q)
    if m:
        take(m)
        filters["keyword"].append(m.group(1).strip())
    for word in TOPIC_WORDS:
        m = re.search(rf"\b{re.escape(word)}\b", q)
        if m and not any(word in k or k in word for k in filters["keyword"]):
            take(m)
            filters["keyword"].append(word)

    # Words no rule used: shown so the user sees what was ignored
    leftover = list(q)
    for a, b in used:
        leftover[a:b] = [" "] * (b - a)
    unread = [w for w in re.findall(r"[a-z0-9$.]+", "".join(leftover)) if w.strip(".") and w not in STOPWORDS]
    return filters, notes, unread


def chip(text):
    return f'<span class="ask-chip">{esc(text)}</span>'


ASK_CSS = """<style>
.radar .ask{border:2px solid #2563a8;border-radius:10px;padding:16px 18px;max-width:760px;background:#fff}
.radar .ask-h{font-size:18px;font-weight:700;color:#1f4f8f}
.radar .ask-tag{display:inline-block;margin-left:10px;font-size:11.5px;font-weight:600;color:#8a4b08;background:#fff4e0;border:1px solid #f0c98a;border-radius:10px;padding:2px 8px;vertical-align:middle}
.radar .bubble{border:1px solid #d9dee5;border-radius:8px;padding:10px 12px;margin:12px 0 4px 80px;background:#fafbfc}
.radar .read{border:1px solid #9db4d6;background:#f3f7fc;border-radius:8px;padding:12px 14px;margin-top:10px}
.radar .ask-chip{display:inline-block;border:1px solid #9db4d6;color:#1f4f8f;background:#fff;border-radius:14px;padding:3px 12px;margin:6px 6px 0 0;font-size:13px}
.radar .note{font-style:italic;color:#4b5563;font-size:12.5px;margin-top:8px}
.radar .hit{padding:8px 0;border-bottom:1px solid #e1e6ec}
.radar .did{border:1px dashed #9fb0c4;border-radius:8px;padding:10px 12px;margin-top:10px;color:#4b5563;font-size:12.5px}
</style>"""


def chips_html(question, filters, notes, unread, after=""):
    chips = [chip(f"Tier: {', '.join(map(str, filters['tier']))}")] if filters["tier"] else []
    chips += [chip(f"Source: {SOURCE_LABELS[s]}") for s in filters["source"]]
    chips += [chip(f"Agency: {name}") for name, _ in filters["agency"]]
    chips += [chip(f"Window: {filters['window']['label']}")] if filters["window"] else []
    chips += [chip(f"Keyword: {k}") for k in filters["keyword"]]
    if not chips:
        chips = ['<span class="none">No filters read from that question. Running it would return every signal.</span>']
    extra = "".join(f'<div class="note">{esc(n)}</div>' for n in notes)
    if unread:
        extra += f'<div class="note">Words I did not use: {esc(", ".join(unread))}</div>'
    return (f'<div class="ask"><div class="ask-h">&#10022; Ask the Radar<span class="ask-tag">{ASK_LABEL}</span></div>'
            f'<div class="bubble">{esc(question)}</div>'
            f'<div class="read"><b>I read that as this filter. Check it before I run it:</b><br>{"".join(chips)}{extra}</div>'
            f'{after}</div>')


def run_filter(df, filters):
    """Apply the chips to the ranked table. Returns matches and a note on anything left out."""
    mask = pd.Series(True, index=df.index)
    left_out = ""
    if filters["tier"]:
        mask &= df["tier"].isin(filters["tier"])
    if filters["source"]:
        mask &= df["source"].isin(filters["source"])
    if filters["agency"]:
        pattern = "|".join(re.escape(needle) for _, needle in filters["agency"])
        mask &= df["agency"].fillna("").str.upper().str.contains(pattern, regex=True)
    if filters["window"]:
        undated = int((mask & df["date"].isna()).sum())
        mask &= df["date"] >= pd.Timestamp(filters["window"]["start"])
        if undated:
            left_out = (f"{undated} otherwise-matching signal{'s have' if undated > 1 else ' has'} no date, so the "
                        f"window could not be checked and {'they are' if undated > 1 else 'it is'} left out.")
    if filters["keyword"]:
        haystack = (df["title"].fillna("") + " " + df["text"].fillna("")).str.lower()
        mask &= haystack.str.contains("|".join(re.escape(k.lower()) for k in filters["keyword"]), regex=True)
    return df[mask], left_out


def results_html(question, filters, notes, unread, matches, left_out, limit=10):
    rows = "".join(
        f'<div class="hit">{badge(r["tier"])} <b>{esc(r["title"])}</b><br>'
        f'<span class="muted">{esc(r["agency"])} · {SOURCE_LABELS[r["source"]]} · {fmt_date(r["date"])} · '
        f'row {i}, open with DETAIL_ROW = {i}</span></div>'
        for i, r in matches.head(limit).iterrows())
    more = f'<div class="muted" style="margin-top:6px">+ {len(matches) - limit} more</div>' if len(matches) > limit else ""
    body = rows or '<div class="none">No signals match. Try removing a chip.</div>'
    gap = f'<div class="note">{esc(left_out)}</div>' if left_out else ""
    count = f"{len(matches)} signal{'s match' if len(matches) != 1 else ' matches'}"
    after = (f'<div class="read"><b>{count}, strongest evidence first:</b>{body}{more}{gap}'
             f'<div class="note">Every item is a record in the feed. Nothing here was generated.</div></div>'
             f'<div class="did"><b>What I did:</b> turned your question into one filter with fixed rules and ran it on '
             f'this run\'s {len(ranked)} signals. I did not read documents, score anything, or add signals.</div>')
    return chips_html(question, filters, notes, unread, after=after)

In [12]:
# Step 1 and 2: read the question and show the chips. Nothing runs yet.
QUESTION = "Tier 1 signals from the last 30 days"

filters, notes, unread = parse_question(QUESTION)
display(HTML(CSS + ASK_CSS + f'<div class="radar">{chips_html(QUESTION, filters, notes, unread)}</div>'))

In [13]:
# Step 3: run the filter. To correct a chip first, edit it here, for example:
# filters["tier"] = [1, 2]        filters["keyword"] = ["zero trust"]        filters["window"] = None
matches, left_out = run_filter(ranked, filters)
ask_page = results_html(QUESTION, filters, notes, unread, matches, left_out)
display(HTML(CSS + ASK_CSS + f'<div class="radar">{ask_page}</div>'))

(MOCKUP_DIR / "ask_the_radar.html").write_text(
    f'<!doctype html><html lang="en"><head><meta charset="utf-8"><title>Ask the Radar: stand-in</title>{CSS}{ASK_CSS}</head>'
    f'<body style="margin:0"><div class="radar">{header("Feed")}<div class="wrap">{ask_page}</div></div></body></html>',
    encoding="utf-8")
print(f"Saved {MOCKUP_DIR / 'ask_the_radar.html'}")

Saved mockup\ask_the_radar.html


In [14]:
# Try other questions: each line shows how the rules read it, without running anything.
for q in ["zero trust awards from the Air Force in the last year",
          "new SAM.gov opportunities from DISA",
          "tier 1 or 2 rules about CMMC",
          "what's new from DoD on zero trust in the last 30 days that has money behind it?"]:
    f, n, u = parse_question(q)
    readable = {k: v for k, v in f.items() if v}
    if readable.get("window"):
        readable["window"] = readable["window"]["label"]
    if readable.get("agency"):
        readable["agency"] = [name for name, _ in readable["agency"]]
    print(f"{q}\n  chips: {readable}\n  notes: {n}\n  not used: {u}\n")

zero trust awards from the Air Force in the last year
  chips: {'source': ['usaspending'], 'agency': ['Air Force'], 'window': 'last year (Oct 11, 2025 to Oct 11, 2026)', 'keyword': ['zero trust']}
  notes: ['"awards" became Source: USAspending.']
  not used: []

new SAM.gov opportunities from DISA
  chips: {'source': ['sam'], 'agency': ['DISA'], 'window': 'last 30 days (Sep 11, 2026 to Oct 11, 2026)'}
  notes: ['"new" became the last 30 days.']
  not used: []

tier 1 or 2 rules about CMMC
  chips: {'tier': [1, 2], 'source': ['federal_register'], 'keyword': ['cmmc']}
  notes: ['"rules" became Source: Federal Register.']
  not used: []

what's new from DoD on zero trust in the last 30 days that has money behind it?
  chips: {'window': 'last 30 days (Sep 11, 2026 to Oct 11, 2026)', 'keyword': ['zero trust']}
  notes: ['"dod" added no filter: every signal in this run is already DoD.', 'DoD awards post on a 90-day delay, so a window this short will miss recent awards.']
  not used: ['money'

## 6. Disagree with a tier (S5, Flow B)

A capture manager can flag any signal by picking one reason from a fixed list. A fixed list keeps the log countable. "Other" requires a one-line note.

Each flag is appended to `data/feedback/disagree_log.csv` with the signal ID, its tier and rule at the time of the flag, the reason, the note, and a timestamp. **Nothing here changes a tier.** The function checks that every tier is the same before and after it writes, and the log waits for human review at the next test session.

Use the picker below, or call `disagree(row, reason, note)` directly. `rows("word")` finds row numbers by title.

In [15]:
import csv

FEEDBACK_DIR = Path("data/feedback")
FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)
DISAGREE_LOG = FEEDBACK_DIR / "disagree_log.csv"
LOG_FIELDS = ["logged_at", "signal_id", "source", "title", "tier_at_flag", "tier_rule", "reason", "note", "status"]

# The fixed reason list. Keys are short names; labels are what the user sees.
DISAGREE_REASONS = {
    "wrong_tier": "Wrong tier",
    "wrong_agency": "Wrong agency or component",
    "duplicate": "Duplicate of another signal",
    "not_relevant": "Not relevant to zero trust work",
    "other": "Other (one-line note required)",
}


def rows(word=""):
    """Row numbers, tiers, and titles for signals whose title contains the word."""
    hits = ranked[ranked["title"].str.contains(word, case=False, na=False)]
    return hits[["tier", "source", "date", "title"]]


def resolve_reason(reason):
    """Accept a key, a label, or a number 1-5 from the list."""
    keys = list(DISAGREE_REASONS)
    text = str(reason).strip().lower()
    if text.isdigit() and 1 <= int(text) <= len(keys):
        return keys[int(text) - 1]
    for key, label in DISAGREE_REASONS.items():
        if text in (key, label.lower(), label.lower().split(" (")[0]):
            return key
    raise ValueError(f"Reason must come from the fixed list: {list(DISAGREE_REASONS.values())}")


def disagree(signal, reason, note=""):
    """Log one disagreement. `signal` is a row number in the ranked table or a signal ID."""
    if isinstance(signal, (int, np.integer)) and signal in ranked.index:
        row = ranked.loc[signal]
    else:
        match = ranked[ranked["id"] == str(signal)]
        if match.empty:
            raise ValueError(f"No signal with row number or ID {signal!r}")
        row = match.iloc[0]

    key = resolve_reason(reason)
    note = " ".join(str(note or "").split())[:200]      # one line, capped
    if key == "other" and not note:
        raise ValueError("'Other' needs a one-line note saying what is wrong.")

    tiers_before = ranked["tier"].copy()
    entry = {
        "logged_at": now_utc(),
        "signal_id": row["id"],
        "source": row["source"],
        "title": row["title"],
        "tier_at_flag": int(row["tier"]),
        "tier_rule": row["tier_rule"],
        "reason": DISAGREE_REASONS[key].split(" (")[0],
        "note": note,
        "status": "logged for review",
    }
    new_file = not DISAGREE_LOG.exists()
    with DISAGREE_LOG.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(entry)

    # The guarantee: logging never changes a score.
    assert ranked["tier"].equals(tiers_before), "A tier changed while logging feedback"

    display(HTML(CSS + ASK_CSS + f"""<div class="radar"><div class="ask" style="border-color:#4a6585">
      <div class="ask-h" style="color:#1d2733">Disagreement logged</div>
      <div style="margin-top:8px">{badge(row["tier"])} <b>{esc(row["title"])}</b></div>
      <div class="muted" style="margin-top:4px">Signal {esc(row["id"])} · rule {esc(row["tier_rule"])}</div>
      <div class="read">Reason: <b>{esc(entry["reason"])}</b>{f"<br>Note: {esc(note)}" if note else ""}</div>
      <div class="did">Logged to {DISAGREE_LOG} at {fmt_ts(entry["logged_at"])} and reviewed at the next test session.
        The tier stays Tier {int(row["tier"])}. Your feedback never changes a score automatically.</div>
    </div></div>"""))
    return entry


def log_summary():
    if not DISAGREE_LOG.exists():
        print("No disagreements logged yet.")
        return None
    log = pd.read_csv(DISAGREE_LOG)
    print(f"{len(log)} disagreements logged. By reason:")
    print(log["reason"].value_counts().to_string())
    return log.tail(5)

In [16]:
# The picker: choose a signal and one reason from the fixed list, then log it.
try:
    import ipywidgets as widgets

    if ranked.empty:
        print("No signals in this run to disagree with.")
    else:
        pick_signal = widgets.Dropdown(
            options=[(f"Row {i} · Tier {r['tier']} · {str(r['title'])[:80]}", i) for i, r in ranked.iterrows()],
            description="Signal", layout=widgets.Layout(width="760px"))
        pick_reason = widgets.Dropdown(
            options=[(label, key) for key, label in DISAGREE_REASONS.items()],
            description="Reason", layout=widgets.Layout(width="420px"))
        note_box = widgets.Text(placeholder="Optional, one line. Required for Other.",
                                description="Note", layout=widgets.Layout(width="760px"))
        log_button = widgets.Button(description="Log disagreement", button_style="primary")
        picker_out = widgets.Output()

        def on_log(_):
            with picker_out:
                picker_out.clear_output()
                try:
                    disagree(pick_signal.value, pick_reason.value, note_box.value)
                except ValueError as err:
                    print(err)

        log_button.on_click(on_log)
        display(widgets.VBox([
            widgets.HTML("<b>Disagree with the tier</b> · logged for review, never applied automatically"),
            pick_signal, pick_reason, note_box, log_button, picker_out]))
except ImportError:
    print("The picker needs ipywidgets (conda install -n radar -c conda-forge ipywidgets). "
          "disagree() in the next cell works without it.")

In [17]:
# Or log by code. Example, left commented so Run All does not write a flag:
# disagree(rows("privacy").index[0], "not_relevant", "Privacy Act notice that only matched on the word cybersecurity")

log_summary()

No disagreements logged yet.


## 7. Save and run summary

Writes the ranked, tiered table to `data/processed/signals.csv` and prints what this run covered, including any source that was skipped.

In [18]:
out_path = PROCESSED_DIR / "signals.csv"
ranked.to_csv(out_path, index=False)

print(f"Saved {len(ranked)} signals to {out_path}")
print(f"Run date {TODAY}, signal window {START} to {TODAY}, award window {AWARD_START} to {TODAY}")
print(f"By tier: {ranked['tier'].value_counts().sort_index().to_dict()}")
print(f"By source: {ranked['source'].value_counts().to_dict()}")
print(f"Blank dates kept blank: {int(ranked['date'].isna().sum())}")
skipped = [s["source"] for s in STATUS if s["status"] != "OK"]
print(f"Skipped sources: {', '.join(skipped) if skipped else 'none'}")

Saved 35 signals to data\processed\signals.csv
Run date 2026-10-11, signal window 2026-07-13 to 2026-10-11, award window 2025-10-11 to 2026-10-11
By tier: {1: 2, 2: 33}
By source: {'usaspending': 30, 'federal_register': 5}
Blank dates kept blank: 0
Skipped sources: none
